# Agregação de Matriz OD por Zoneamento Customizado

Este notebook consolida a rotina completa para agregar a matriz OD de municípios (`vw_mtx_pessoas_auto_calib`) em qualquer zoneamento customizado, tratando dois cenários:

1. **Agregação simples** — quando cada município está 100% contido em uma única zona do novo zoneamento.
2. **Desagregação por setor censitário** — quando um município é cortado por mais de uma zona, as viagens são distribuídas proporcionalmente ao peso dos setores censitários (`cd_sit`) em cada zona. Peso = `1/cd_sit` (setor mais denso pesa mais).

## Estrutura das tabelas envolvidas

| Schema | Tabela | Papel |
|---|---|---|
| `VISUM` | `vw_mtx_pessoas_auto_calib` | Matriz OD por município (colunas `o`, `d`, `viagens`) |
| `VISUM` | `tbl_shp_municipiosibge_2024` | Shapefile municípios IBGE (`cd_mun`, `wkb_geometry`) |
| `VISUM` | `tbl_shp_pequi` (exemplo) | Shapefile do zoneamento novo (`id`, `geom`) |
| `TERRITORIO` | `tbl_zonas_censitarias_2022` | Shapefile de setores censitários (`cd_sit`, geom) |

## Observações importantes

- Os SRIDs diferem entre tabelas: IBGE em **4674** (SIRGAS 2000), zoneamentos em **4326** (WGS 84). Todas as operações espaciais fazem `ST_Transform` quando necessário.
- Os nomes de schema (`VISUM`, `TERRITORIO`) foram criados com letras maiúsculas, então aparecem sempre entre aspas duplas.
- A rotina é **parametrizada**: você passa o nome da tabela de zoneamento e ela faz todo o pipeline.

In [ ]:
import os
import psycopg2
import pandas as pd
from dotenv import load_dotenv

# Carrega o .env do projeto python_secrets_lib
env_path = r"C:\Users\zeno.filho\projetos\python_secrets_lib\.env"
load_dotenv(dotenv_path=env_path)


def get_db_connection():
    """Retorna uma conexão psycopg2 lendo credenciais do .env."""
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT", 5432),
    )


def run_sql(sql, params=None, fetch=False):
    """Executa um SQL. Se fetch=True, retorna DataFrame com o resultado."""
    conn = get_db_connection()
    try:
        if fetch:
            df = pd.read_sql(sql, conn, params=params)
            return df
        else:
            cur = conn.cursor()
            cur.execute(sql, params)
            conn.commit()
            cur.close()
    finally:
        conn.close()


# Teste rápido da conexão
df_teste = run_sql("SELECT current_database(), current_user, version();", fetch=True)
print(df_teste.to_string(index=False))

## 1. Parâmetros do zoneamento

Edite aqui para trocar de zoneamento sem mexer no resto do notebook.

In [ ]:
# Schema e tabela do novo zoneamento
ZON_SCHEMA = "VISUM"
ZON_TABELA = "tbl_shp_pequi"       # tem maiúsculas -> precisa de aspas
ZON_COL_ID = "id"                  # coluna identificadora da zona
ZON_COL_GEOM = "geom"              # coluna de geometria
ZON_SRID = 4326                    # SRID da tabela de zoneamento

# Sufixo usado nas tabelas de saída (ex.: de_para_mun_pequi, mtx_od_pequi)
SUFIXO = "pequi"

# Matriz OD de entrada
OD_SCHEMA = "VISUM"
OD_VIEW = "vw_mtx_pessoas_auto_calib"
OD_COL_O = "o"
OD_COL_D = "d"
OD_COL_VIAGENS = "viagens"

# Municípios
MUN_SCHEMA = "VISUM"
MUN_TABELA = "tbl_shp_municipiosibge_2024"
MUN_COL_CD = "cd_mun"
MUN_COL_GEOM = "wkb_geometry"
MUN_SRID = 4674

# Setores censitários
SET_SCHEMA = "TERRITORIO"
SET_TABELA = "tbl_zonas_censitarias_2022"
SET_COL_CDSIT = "cd_sit"
SET_COL_CDMUN = "cd_mun"            # ajuste se o nome da coluna for diferente
SET_COL_GEOM = "geom"               # ajuste se for wkb_geometry
SET_SRID = 4674                     # ajuste se diferente

print(f"Zoneamento atual: {ZON_SCHEMA}.{ZON_TABELA} (sufixo: {SUFIXO})")

## 2. Verificações preliminares

Antes de rodar a rotina, confirme os SRIDs e a estrutura das tabelas. Ajuste os parâmetros da célula anterior se algo não bater.

In [ ]:
sql_srids = f'''
SELECT
    (SELECT DISTINCT ST_SRID({MUN_COL_GEOM}) FROM "{MUN_SCHEMA}"."{MUN_TABELA}")    AS srid_municipios,
    (SELECT DISTINCT ST_SRID({ZON_COL_GEOM}) FROM "{ZON_SCHEMA}"."{ZON_TABELA}")    AS srid_zoneamento,
    (SELECT DISTINCT ST_SRID({SET_COL_GEOM}) FROM "{SET_SCHEMA}"."{SET_TABELA}")    AS srid_setores
'''
run_sql(sql_srids, fetch=True)

In [ ]:
# Verifica tipos das colunas de junção (o/d vs cd_mun) - sensível a mismatch int/varchar
sql_tipos = '''
SELECT table_schema, table_name, column_name, data_type
FROM information_schema.columns
WHERE (table_schema, table_name, column_name) IN (
    (%s, %s, %s),
    (%s, %s, %s),
    (%s, %s, %s)
)
ORDER BY table_schema, table_name, column_name;
'''
params = (
    MUN_SCHEMA, MUN_TABELA, MUN_COL_CD,
    OD_SCHEMA, OD_VIEW, OD_COL_O,
    OD_SCHEMA, OD_VIEW, OD_COL_D,
)
run_sql(sql_tipos, params=params, fetch=True)

## 3. De-para município → zona

Gera duas tabelas:

- **`de_para_mun_<sufixo>`** — todos os pares (município, zona) com a área de interseção. Um município que toca N zonas aparece N vezes.
- **`mun_status_<sufixo>`** — marca cada município como `inteiro` (contido em uma única zona) ou `dividido` (cortado por 2+).

Usamos `ST_PointOnSurface` sobre o centroide do município para decidir a zona "principal" quando ele está inteiro. Para os divididos, a desagregação por setor entra em ação na etapa 4.

In [ ]:
sql_depara = f'''
DROP TABLE IF EXISTS "{ZON_SCHEMA}".de_para_mun_{SUFIXO};
DROP TABLE IF EXISTS "{ZON_SCHEMA}".mun_status_{SUFIXO};

-- Todas as interseções município x zona (área em m² na projeção geográfica original)
CREATE TABLE "{ZON_SCHEMA}".de_para_mun_{SUFIXO} AS
SELECT
    m.{MUN_COL_CD}::text AS cd_mun,
    z.{ZON_COL_ID}       AS zona_id,
    ST_Area(
        ST_Intersection(
            ST_Transform(m.{MUN_COL_GEOM}, {ZON_SRID}),
            z.{ZON_COL_GEOM}
        )
    ) AS area_intersec
FROM "{MUN_SCHEMA}"."{MUN_TABELA}" m
JOIN "{ZON_SCHEMA}"."{ZON_TABELA}" z
  ON ST_Intersects(
       ST_Transform(m.{MUN_COL_GEOM}, {ZON_SRID}),
       z.{ZON_COL_GEOM}
     )
WHERE ST_Area(
        ST_Intersection(
            ST_Transform(m.{MUN_COL_GEOM}, {ZON_SRID}),
            z.{ZON_COL_GEOM}
        )
      ) > 0;

CREATE INDEX idx_depara_{SUFIXO}_cdmun ON "{ZON_SCHEMA}".de_para_mun_{SUFIXO}(cd_mun);
CREATE INDEX idx_depara_{SUFIXO}_zona  ON "{ZON_SCHEMA}".de_para_mun_{SUFIXO}(zona_id);

-- Status: município inteiro ou dividido
CREATE TABLE "{ZON_SCHEMA}".mun_status_{SUFIXO} AS
SELECT
    cd_mun,
    COUNT(*) AS n_zonas_tocadas,
    CASE WHEN COUNT(*) = 1 THEN 'inteiro' ELSE 'dividido' END AS status
FROM "{ZON_SCHEMA}".de_para_mun_{SUFIXO}
GROUP BY cd_mun;

CREATE INDEX idx_status_{SUFIXO}_cdmun ON "{ZON_SCHEMA}".mun_status_{SUFIXO}(cd_mun);
'''
run_sql(sql_depara)
print("De-para e status criados.")

In [ ]:
# Resumo: quantos municípios inteiros x divididos
sql_resumo = f'''
SELECT status, COUNT(*) AS n_municipios
FROM "{ZON_SCHEMA}".mun_status_{SUFIXO}
GROUP BY status
ORDER BY status;
'''
run_sql(sql_resumo, fetch=True)

## 4. Pesos por setor censitário

Para cada município **dividido**, calculamos o peso de cada zona nova como:

$$
\text{peso}_{(\text{mun}, \text{zona})} = \sum_{\text{setor} \in \text{mun} \cap \text{zona}} \frac{1}{\text{cd\_sit}_{\text{setor}}}
$$

A fração das viagens do município que vai para cada zona é esse peso normalizado pelo peso total do município.

**Como decidimos em qual zona um setor está?** Usamos o centroide (`ST_PointOnSurface`) do setor contra o polígono da zona — setor inteiro cai em uma zona só, coerente com a premissa que você já usa para municípios inteiros.

**Atenção:** essa etapa só considera setores dos municípios divididos. Os inteiros não precisam de pesagem.

In [ ]:
sql_pesos = f'''
DROP TABLE IF EXISTS "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO};

CREATE TABLE "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO} AS
WITH setores_muni_dividido AS (
    -- Só olha setores dos municípios que precisam de desagregação
    SELECT s.*
    FROM "{SET_SCHEMA}"."{SET_TABELA}" s
    JOIN "{ZON_SCHEMA}".mun_status_{SUFIXO} ms
      ON s.{SET_COL_CDMUN}::text = ms.cd_mun
    WHERE ms.status = 'dividido'
),
setor_para_zona AS (
    -- Cada setor é atribuído à zona nova que contém seu centroide
    SELECT
        s.{SET_COL_CDMUN}::text AS cd_mun,
        z.{ZON_COL_ID}          AS zona_id,
        s.{SET_COL_CDSIT}       AS cd_sit,
        (1.0 / NULLIF(s.{SET_COL_CDSIT}::numeric, 0)) AS peso_setor
    FROM setores_muni_dividido s
    JOIN "{ZON_SCHEMA}"."{ZON_TABELA}" z
      ON ST_Contains(
           z.{ZON_COL_GEOM},
           ST_Transform(ST_PointOnSurface(s.{SET_COL_GEOM}), {ZON_SRID})
         )
)
SELECT
    cd_mun,
    zona_id,
    SUM(peso_setor) AS peso_zona,
    COUNT(*)        AS n_setores
FROM setor_para_zona
GROUP BY cd_mun, zona_id;

-- Fração normalizada por município
ALTER TABLE "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}
  ADD COLUMN fracao numeric;

UPDATE "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO} p
SET fracao = p.peso_zona / t.peso_total
FROM (
    SELECT cd_mun, SUM(peso_zona) AS peso_total
    FROM "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}
    GROUP BY cd_mun
) t
WHERE p.cd_mun = t.cd_mun;

CREATE INDEX idx_pesos_{SUFIXO}_cdmun ON "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}(cd_mun);
CREATE INDEX idx_pesos_{SUFIXO}_zona  ON "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}(zona_id);
'''
run_sql(sql_pesos)
print("Pesos município x zona calculados.")

In [ ]:
# Sanidade: a soma das frações por município deve ser 1.0 (ou muito próxima)
sql_check_fracao = f'''
SELECT
    cd_mun,
    ROUND(SUM(fracao)::numeric, 6) AS soma_fracao,
    COUNT(*) AS n_zonas
FROM "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}
GROUP BY cd_mun
HAVING ROUND(SUM(fracao)::numeric, 6) <> 1.0
ORDER BY cd_mun
LIMIT 20;
'''
df_check = run_sql(sql_check_fracao, fetch=True)
if df_check.empty:
    print("✅ Todas as frações somam 1 por município.")
else:
    print("⚠️ Municípios com soma de fração diferente de 1 (pode ser arredondamento):")
    print(df_check)

## 5. Tabela unificada de rateio município → zona

Agora unimos os dois casos em uma única tabela:

- Município **inteiro** → fração = 1.0 para a zona que o contém.
- Município **dividido** → uma linha por zona com a fração calculada na etapa anterior.

Essa tabela é o que vamos usar pra agregar a matriz OD.

In [ ]:
sql_rateio = f'''
DROP TABLE IF EXISTS "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO};

CREATE TABLE "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} AS
-- Caso 1: municípios inteiros (fração = 1)
SELECT
    d.cd_mun,
    d.zona_id,
    1.0::numeric AS fracao
FROM "{ZON_SCHEMA}".de_para_mun_{SUFIXO} d
JOIN "{ZON_SCHEMA}".mun_status_{SUFIXO} ms
  ON d.cd_mun = ms.cd_mun
WHERE ms.status = 'inteiro'

UNION ALL

-- Caso 2: municípios divididos (fração calculada pelos setores)
SELECT
    p.cd_mun,
    p.zona_id,
    p.fracao
FROM "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO} p;

CREATE INDEX idx_rateio_{SUFIXO}_cdmun ON "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO}(cd_mun);
CREATE INDEX idx_rateio_{SUFIXO}_zona  ON "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO}(zona_id);
'''
run_sql(sql_rateio)
print("Tabela de rateio unificada criada.")

In [ ]:
# Verifica cobertura: todo município com OD precisa estar no rateio
sql_cobertura = f'''
WITH mun_na_matriz AS (
    SELECT {OD_COL_O}::text AS cd_mun FROM "{OD_SCHEMA}"."{OD_VIEW}"
    UNION
    SELECT {OD_COL_D}::text AS cd_mun FROM "{OD_SCHEMA}"."{OD_VIEW}"
)
SELECT COUNT(*) AS municipios_sem_rateio
FROM mun_na_matriz m
LEFT JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} r ON m.cd_mun = r.cd_mun
WHERE r.cd_mun IS NULL;
'''
run_sql(sql_cobertura, fetch=True)

## 6. Agregação da matriz OD

Para cada par origem-destino da matriz original, expandimos usando o rateio dos dois lados:

$$
\text{viagens}_{(Z_o, Z_d)} = \sum_{(m_o, m_d)} \text{viagens}_{(m_o, m_d)} \cdot f_{(m_o, Z_o)} \cdot f_{(m_d, Z_d)}
$$

Onde $f$ é a fração do rateio. Para municípios inteiros, $f = 1$, então o comportamento é idêntico ao caso simples.

In [ ]:
sql_agregacao = f'''
DROP TABLE IF EXISTS "{ZON_SCHEMA}".mtx_od_{SUFIXO};

CREATE TABLE "{ZON_SCHEMA}".mtx_od_{SUFIXO} AS
SELECT
    ro.zona_id AS o,
    rd.zona_id AS d,
    SUM(od.{OD_COL_VIAGENS} * ro.fracao * rd.fracao) AS viagens
FROM "{OD_SCHEMA}"."{OD_VIEW}" od
JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} ro ON od.{OD_COL_O}::text = ro.cd_mun
JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} rd ON od.{OD_COL_D}::text = rd.cd_mun
GROUP BY ro.zona_id, rd.zona_id
ORDER BY ro.zona_id, rd.zona_id;

CREATE INDEX idx_mtx_{SUFIXO}_o ON "{ZON_SCHEMA}".mtx_od_{SUFIXO}(o);
CREATE INDEX idx_mtx_{SUFIXO}_d ON "{ZON_SCHEMA}".mtx_od_{SUFIXO}(d);
'''
run_sql(sql_agregacao)
print("Matriz OD agregada criada.")

## 7. Validações

A soma de viagens original tem que ser **praticamente igual** à soma agregada (pode haver diferença mínima de ponto flutuante). Se a diferença for grande, provavelmente há municípios na matriz OD que não estão cobertos pelo zoneamento.

In [ ]:
sql_totais = f'''
SELECT
    (SELECT SUM({OD_COL_VIAGENS}) FROM "{OD_SCHEMA}"."{OD_VIEW}") AS total_original,
    (SELECT SUM(viagens)          FROM "{ZON_SCHEMA}".mtx_od_{SUFIXO}) AS total_agregado
'''
df_totais = run_sql(sql_totais, fetch=True)
df_totais["diff_pct"] = (
    (df_totais["total_agregado"] - df_totais["total_original"])
    / df_totais["total_original"] * 100
)
df_totais

In [ ]:
# Contagem de pares OD antes e depois
sql_pares = f'''
SELECT
    (SELECT COUNT(*) FROM "{OD_SCHEMA}"."{OD_VIEW}")          AS pares_original,
    (SELECT COUNT(*) FROM "{ZON_SCHEMA}".mtx_od_{SUFIXO})     AS pares_agregado
'''
run_sql(sql_pares, fetch=True)

In [ ]:
# Top 10 pares do resultado, como preview
sql_preview = f'''
SELECT o, d, viagens
FROM "{ZON_SCHEMA}".mtx_od_{SUFIXO}
ORDER BY viagens DESC
LIMIT 10;
'''
run_sql(sql_preview, fetch=True)

## 8. Função reutilizável

A função abaixo consolida todas as etapas. Útil quando você precisa testar vários zoneamentos em sequência.

In [ ]:
def agregar_od_por_zoneamento(
    zon_schema, zon_tabela, zon_col_id, zon_col_geom, zon_srid,
    sufixo,
    od_schema="VISUM", od_view="vw_mtx_pessoas_auto_calib",
    od_col_o="o", od_col_d="d", od_col_viagens="viagens",
    mun_schema="VISUM", mun_tabela="tbl_shp_municipiosibge_2024",
    mun_col_cd="cd_mun", mun_col_geom="wkb_geometry",
    set_schema="TERRITORIO", set_tabela="tbl_zonas_censitarias_2022",
    set_col_cdsit="cd_sit", set_col_cdmun="cd_mun", set_col_geom="geom",
):
    """Executa a pipeline completa de agregação/desagregação da matriz OD.

    Cria no schema do zoneamento:
      - de_para_mun_<sufixo>
      - mun_status_<sufixo>
      - pesos_mun_zona_<sufixo>
      - rateio_mun_zona_<sufixo>
      - mtx_od_<sufixo>

    Retorna dict com totais de validação.
    """
    ctx = dict(
        ZS=zon_schema, ZT=zon_tabela, ZID=zon_col_id, ZG=zon_col_geom, ZSRID=zon_srid,
        SF=sufixo,
        OS=od_schema, OV=od_view, OO=od_col_o, OD=od_col_d, OV_=od_col_viagens,
        MS=mun_schema, MT=mun_tabela, MC=mun_col_cd, MG=mun_col_geom,
        SS=set_schema, ST=set_tabela, SC=set_col_cdsit, SM=set_col_cdmun, SG=set_col_geom,
    )

    sql = '''
    DROP TABLE IF EXISTS "{ZS}".de_para_mun_{SF};
    DROP TABLE IF EXISTS "{ZS}".mun_status_{SF};
    DROP TABLE IF EXISTS "{ZS}".pesos_mun_zona_{SF};
    DROP TABLE IF EXISTS "{ZS}".rateio_mun_zona_{SF};
    DROP TABLE IF EXISTS "{ZS}".mtx_od_{SF};

    CREATE TABLE "{ZS}".de_para_mun_{SF} AS
    SELECT m.{MC}::text AS cd_mun, z.{ZID} AS zona_id,
           ST_Area(ST_Intersection(ST_Transform(m.{MG}, {ZSRID}), z.{ZG})) AS area_intersec
    FROM "{MS}"."{MT}" m
    JOIN "{ZS}"."{ZT}" z
      ON ST_Intersects(ST_Transform(m.{MG}, {ZSRID}), z.{ZG})
    WHERE ST_Area(ST_Intersection(ST_Transform(m.{MG}, {ZSRID}), z.{ZG})) > 0;
    CREATE INDEX ON "{ZS}".de_para_mun_{SF}(cd_mun);
    CREATE INDEX ON "{ZS}".de_para_mun_{SF}(zona_id);

    CREATE TABLE "{ZS}".mun_status_{SF} AS
    SELECT cd_mun, COUNT(*) AS n_zonas_tocadas,
           CASE WHEN COUNT(*) = 1 THEN 'inteiro' ELSE 'dividido' END AS status
    FROM "{ZS}".de_para_mun_{SF} GROUP BY cd_mun;
    CREATE INDEX ON "{ZS}".mun_status_{SF}(cd_mun);

    CREATE TABLE "{ZS}".pesos_mun_zona_{SF} AS
    WITH setores_div AS (
        SELECT s.* FROM "{SS}"."{ST}" s
        JOIN "{ZS}".mun_status_{SF} ms ON s.{SM}::text = ms.cd_mun
        WHERE ms.status = 'dividido'
    ),
    sp AS (
        SELECT s.{SM}::text AS cd_mun, z.{ZID} AS zona_id, s.{SC} AS cd_sit,
               (1.0 / NULLIF(s.{SC}::numeric, 0)) AS peso_setor
        FROM setores_div s
        JOIN "{ZS}"."{ZT}" z
          ON ST_Contains(z.{ZG}, ST_Transform(ST_PointOnSurface(s.{SG}), {ZSRID}))
    )
    SELECT cd_mun, zona_id, SUM(peso_setor) AS peso_zona, COUNT(*) AS n_setores
    FROM sp GROUP BY cd_mun, zona_id;

    ALTER TABLE "{ZS}".pesos_mun_zona_{SF} ADD COLUMN fracao numeric;
    UPDATE "{ZS}".pesos_mun_zona_{SF} p
    SET fracao = p.peso_zona / t.peso_total
    FROM (SELECT cd_mun, SUM(peso_zona) AS peso_total
          FROM "{ZS}".pesos_mun_zona_{SF} GROUP BY cd_mun) t
    WHERE p.cd_mun = t.cd_mun;
    CREATE INDEX ON "{ZS}".pesos_mun_zona_{SF}(cd_mun);

    CREATE TABLE "{ZS}".rateio_mun_zona_{SF} AS
    SELECT d.cd_mun, d.zona_id, 1.0::numeric AS fracao
    FROM "{ZS}".de_para_mun_{SF} d
    JOIN "{ZS}".mun_status_{SF} ms ON d.cd_mun = ms.cd_mun
    WHERE ms.status = 'inteiro'
    UNION ALL
    SELECT cd_mun, zona_id, fracao FROM "{ZS}".pesos_mun_zona_{SF};
    CREATE INDEX ON "{ZS}".rateio_mun_zona_{SF}(cd_mun);
    CREATE INDEX ON "{ZS}".rateio_mun_zona_{SF}(zona_id);

    CREATE TABLE "{ZS}".mtx_od_{SF} AS
    SELECT ro.zona_id AS o, rd.zona_id AS d,
           SUM(od.{OV_} * ro.fracao * rd.fracao) AS viagens
    FROM "{OS}"."{OV}" od
    JOIN "{ZS}".rateio_mun_zona_{SF} ro ON od.{OO}::text = ro.cd_mun
    JOIN "{ZS}".rateio_mun_zona_{SF} rd ON od.{OD}::text = rd.cd_mun
    GROUP BY ro.zona_id, rd.zona_id;
    CREATE INDEX ON "{ZS}".mtx_od_{SF}(o);
    CREATE INDEX ON "{ZS}".mtx_od_{SF}(d);
    '''.format(**ctx)

    run_sql(sql)

    sql_val = '''
    SELECT
        (SELECT SUM({OV_}) FROM "{OS}"."{OV}")          AS total_original,
        (SELECT SUM(viagens) FROM "{ZS}".mtx_od_{SF})   AS total_agregado
    '''.format(**ctx)
    df = run_sql(sql_val, fetch=True)
    return df.to_dict(orient="records")[0]


# Exemplo de uso:
# resultado = agregar_od_por_zoneamento(
#     zon_schema="VISUM", zon_tabela="tbl_shp_pequi",
#     zon_col_id="id", zon_col_geom="geom", zon_srid=4326,
#     sufixo="pequi",
# )
# print(resultado)

## Notas finais

- **Performance**: se a matriz for muito grande (milhões de pares), `REFRESH MATERIALIZED VIEW "VISUM".vw_mtx_pessoas_auto_calib` antes da agregação pra garantir dados frescos, e considere `VACUUM ANALYZE` nas tabelas de rateio criadas.
- **SRIDs**: toda operação espacial converte para o SRID do zoneamento (`ZON_SRID`). Se um dia o zoneamento vier em SIRGAS 2000 direto, basta mudar o parâmetro.
- **Peso alternativo**: se mais tarde você quiser experimentar outra função de peso (ex.: população do setor, domicílios), é só trocar a expressão `1.0 / cd_sit` na CTE `sp` da função.
- **Edge case**: se um setor cair fora de qualquer polígono do zoneamento (o que teoricamente não deveria ocorrer), ele some do cálculo de peso. Na prática, para municípios divididos, vale rodar uma validação cruzada contando setores esperados vs pesados.